# POC DeconvNet

- Creation : 24/04/2025


- [ ] A partir d'un modèle de type Convnet charger
- [ ] Créer un modèle de type Deconvnet associé
- [ ] Vérifier la possibilité de le faire fonctionner sur GPU, tout en lui fournissant les indices de switch.

## Modules

In [1]:
from typing import List, Tuple, Optional

import torch
import torch.nn as nn
import torchvision

from utils.convnet_wrapper_for_deconvolution import ConvnetWrapperForDeconvolution
from utils.utils_deconv import make_coherent_before_max_unpool2d
from utils.clean_map import clean_feature_maps

## Device

In [2]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"{device} est disponible")

mps est disponible


## Modèle de type Convnet

In [3]:
model_name = "alexnet"
TORCHVISION_MODELS_WEIGHTS = torchvision.models.AlexNet_Weights
model_convnet = torch.hub.load('pytorch/vision', model_name, weights=TORCHVISION_MODELS_WEIGHTS.IMAGENET1K_V1)
model_convnet.eval()

Using cache found in /Users/me/.cache/torch/hub/pytorch_vision_main


AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

In [4]:
model_for_deconv = ConvnetWrapperForDeconvolution(model_convnet, model_convnet.features)

## Définition classe DeconvNet


In [5]:
class Sub(nn.Module):
    def __init__(self, tensor_to_sub: torch.Tensor) -> None:
        super().__init__()
        self.tensor_to_sub = tensor_to_sub

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x - self.tensor_to_sub
    
    def string(self) -> str:
        return f"Sub({self.tensor_to_sub})"

In [6]:
class Deconvnet(nn.Module):
    def __init__(self,
                 wrapped_convnet: ConvnetWrapperForDeconvolution,
                 flip_kernels: bool = False,
                 use_bias: bool = False
                 ) -> None:    
        super().__init__()
        self.wrapped_convnet = wrapped_convnet

        self.deconv_model = torch.nn.Sequential()
        self.flip_kernels = flip_kernels
        self.use_bias = use_bias
        self.build_deconvnet()
    
    def build_deconvnet(self):
        with torch.no_grad():
            for module in reversed(self.wrapped_convnet.convnet_features):
                if isinstance(module, nn.MaxPool2d):
                    self.deconv_model.append(nn.MaxUnpool2d(
                        kernel_size=module.kernel_size,
                        stride=module.stride,
                        padding=module.padding
                    ))
                elif isinstance(module, nn.ReLU):
                    self.deconv_model.append(nn.ReLU())
                elif isinstance(module, nn.Conv2d):

                    if self.use_bias and module.bias != None:
                        bias = module.bias.clone().to("cpu").unsqueeze(dim=0).unsqueeze(dim=0).reshape(module.bias.size(0), 1, 1).unsqueeze(dim=0)
                        self.deconv_model.append(Sub(bias))

                    conv = nn.ConvTranspose2d(
                        in_channels=module.out_channels,
                        out_channels=module.in_channels,
                        kernel_size=module.kernel_size,
                        stride=module.stride,
                        padding=module.padding,
                        output_padding=1 if module.stride[0] > 1 else 0, # Because stride > 1
                        dilation=module.dilation,
                        bias=False
                    )

                    weight = module.weight.clone().to("cpu")
                    conv.weight.copy_(torch.flip(weight, [2, 3]) if self.flip_kernels else weight)
                    self.deconv_model.append(conv)

    def forward(self, x, from_idx_layer, switch_indices: List[Tuple[int, torch.Tensor]], verbose: bool = False):
        """
        x: input tensor
        switch_indices: list of tuples (index, indices) where index is the index of the layer and indices are the indices to be switched
        """
        print("*************************")
        
        if from_idx_layer < 0:
            from_idx_layer += len(self.deconv_model)
        assert from_idx_layer < len(self.deconv_model), f"from_idx_layer must be in range [-{len(self.deconv_model)}, {len(self.deconv_model)}["

        from_idx_layer = len(self.deconv_model) - from_idx_layer
        for i, module in enumerate(self.deconv_model):
            if i < from_idx_layer:
                continue
            if verbose:
                print(f"[{i}] : ", module)
            if isinstance(module, nn.MaxUnpool2d):
                _, indices = switch_indices.pop()
                indices = make_coherent_before_max_unpool2d(x, indices)
                x = module(x, indices)
            else:
                x = module(x)
            if verbose:
                print(f"\t\t> output.size :{x.size()} | min : {x.min().item():.3f} | max : {x.max().item()}")
        return x

In [7]:
deconvnet = Deconvnet(model_for_deconv, flip_kernels=False, use_bias=False)
print(deconvnet)

Deconvnet(
  (deconv_model): Sequential(
    (0): MaxUnpool2d(kernel_size=(3, 3), stride=(2, 2), padding=(0, 0))
    (1): ReLU()
    (2): ConvTranspose2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (3): ReLU()
    (4): ConvTranspose2d(256, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (5): ReLU()
    (6): ConvTranspose2d(384, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (7): MaxUnpool2d(kernel_size=(3, 3), stride=(2, 2), padding=(0, 0))
    (8): ReLU()
    (9): ConvTranspose2d(192, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2), bias=False)
    (10): MaxUnpool2d(kernel_size=(3, 3), stride=(2, 2), padding=(0, 0))
    (11): ReLU()
    (12): ConvTranspose2d(64, 3, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2), output_padding=(1, 1), bias=False)
  )
)


## Deconvolution avec DeconvNet

In [8]:
def perform_deconvolution(
        x_to_deconv: torch.Tensor,
        deconvnet: Deconvnet,
        idx_layer: int,
        switch_indices: List[Tuple[int, torch.Tensor]] = None,
        verbose: bool = False,
        ) -> torch.Tensor:
    
    return deconvnet.forward(
        x_to_deconv,
        idx_layer,
        switch_indices,
        verbose=verbose
    )

def deconvolution(
        x: torch.Tensor,
        deconvnet: Deconvnet,
        idx_layer: int = None,
        clean_feature_map: bool = True,
        idx_map: Optional[int] = None,
        pos: Optional[Tuple[int, int]] = None,
        return_pos: bool = True,
        verbose: bool = False
        ) -> torch.Tensor|Tuple[torch.Tensor, torch.Tensor]:
    """
    Assuming than conv_model have a features composant corresponding to CNN part.

    Args:
        - cnn_model
        - x (torch.Tensor) : input to forward until idx_layer module (size [B, C, H, W])
        - idx_layer
        - flip_kernels
        - use_bias
        - clean_feature_map
        - idx_map (int) : used if clean_feature_map. Chanel index. If False, it returns a value by item. If None, return a max value by channel.
        - pos ((int, int), optional) : used if clean_feature_map. indice of specific activation in any map of feature_maps.
        - return_pos : used if clean_feature_map. return index in feature_maps of kept activations.
        
        - verbose

    Returns:
        - backward deconvolution result
        - coords of max activation, required for receptive field process, if clean_feature_map
    """
    convnet_features = deconvnet.wrapped_convnet.convnet_features

    # Normalizing idx_layer
    if idx_layer == None:
        idx_layer = len(convnet_features) - 1
    elif idx_layer < 0:
        idx_layer += len(convnet_features)
    # check if idx_layer is coherent with cnn_features
    assert (0 <= idx_layer) and (idx_layer < len(convnet_features)), \
        f"i should be in [-{len(convnet_features)}; {len(convnet_features)}["
    
    # Generate feature maps and switch indices
    output, switch_indices = deconvnet.wrapped_convnet.forward_for_deconv(
        x, idx_layer, return_switch_indices=True, verbose=verbose
        )
    
    if verbose:
        print(f"forwarded output size {output.size()} | switch indices : {len(switch_indices)}", end="")
        print(f" | min : {output.min().item()} | max : {output.max().item()}")

    # Clean idx_map feature maps
    if clean_feature_map:
        pool_indices = None

        # si le dernier module est un pooling, afficher la taille des indices de pooling qu'il a généré
        if isinstance(convnet_features[idx_layer], nn.MaxPool2d):
            _, pool_indices = switch_indices[-1]
            if verbose:
                print("pool_indices.size", pool_indices.size())

        # Nettoyage de la carte des caractéristiques qu'est la sortie
        cleaned = clean_feature_maps(
            output, idx_map=idx_map, pos=pos, pool_indices=pool_indices,
            return_pos=return_pos, keep_only_last_occurrence=False
            )
        
        #if verbose:
        #    print("cleaned", cleaned)

        if return_pos:
            output, pos = cleaned
        else:
            output = cleaned

    # perform deconvnet
    deconv = perform_deconvolution(
        output,
        deconvnet,
        idx_layer,
        switch_indices,
        verbose=verbose,
        )

    if return_pos and clean_feature_map:
        return deconv, pos
    else:
        return deconv

## Convolution puis déconvolution

### Données

In [9]:
import os
from datasets import DATASET_1, CustomImageDataset, get_label_data_from_filename
from torchvision import transforms as T
from torch.utils.data import DataLoader

DATASET = DATASET_1 # le 50K images

imagenet_mean = DATASET["means"]
imagenet_std = DATASET["stds"]

geo_transforms = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
])

transforms = TORCHVISION_MODELS_WEIGHTS.IMAGENET1K_V1.transforms()

get_label_data = lambda f: get_label_data_from_filename(f, DATASET["path"])
dataset_path = DATASET["mounted_path"] if os.path.exists(DATASET["mounted_path"]) else DATASET["path"]
print(f"Utilisation du dataset {DATASET['name']} situé dans {dataset_path}")

dataset = CustomImageDataset(
    dataset_path,
    transform=transforms,
    extension="JPEG",
    dataset_mode=False,
    only_label_idx=False, # On a besoin de l'index pour le nom du fichier
    get_label_data=get_label_data,
    )

batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

Utilisation du dataset imagenet_val_images (50K images) situé dans /Users/me/Documents/Work/Dev/_data/imagenet_val_images


### Activations

In [10]:
N_images = 10
coord_activations = {7: [(327, 1, 0), (379, 4, 3)]}
model_for_deconv.to(device)
all_activations = []
for i in range(0, N_images):
    image, image_trfm, label_idx, label_code, input_file_idx = dataset[i]
    batch_input = image_trfm.unsqueeze(dim=0).to(device)
    activations = model_for_deconv.get_activations(batch_input, coord_activations, verbose=False)
    all_activations.append((activations, torch.tensor([input_file_idx])))

print(f"Toutes les activations : {all_activations}")

Toutes les activations : [({7: [tensor([0.]), tensor([0.])]}, tensor([0])), ({7: [tensor([0.]), tensor([3.8228])]}, tensor([1])), ({7: [tensor([0.]), tensor([0.])]}, tensor([2])), ({7: [tensor([4.6041]), tensor([0.])]}, tensor([3])), ({7: [tensor([0.]), tensor([0.])]}, tensor([4])), ({7: [tensor([2.9854]), tensor([0.])]}, tensor([5])), ({7: [tensor([0.]), tensor([10.7379])]}, tensor([6])), ({7: [tensor([0.]), tensor([0.])]}, tensor([7])), ({7: [tensor([0.2031]), tensor([11.5071])]}, tensor([8])), ({7: [tensor([0.]), tensor([0.])]}, tensor([9]))]


### Deconv

In [11]:
idx_layer = 7
coord = coord_activations[idx_layer][1]

i_image = 8
x = all_activations[i_image][0]
print(x[idx_layer][1])

image, image_trfm, label_idx, label_code, input_file_idx = dataset[i_image]
idx_map, row, col = coord
batch_input = image_trfm.unsqueeze(dim=0).to(device)

tensor([11.5071])


In [12]:
deconvnet.to(device)
output_deconv = deconvolution(
    batch_input,
    deconvnet,
    idx_layer=idx_layer,
    idx_map=idx_map,
    pos=(row, col),
    clean_feature_map=True,
    return_pos=False,
    verbose=True
    ).detach().cpu()

[0] forward  Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
	 ouput size: torch.Size([1, 64, 55, 55])
[1] forward  ReLU(inplace=True)
	 ouput size: torch.Size([1, 64, 55, 55])
[2] forward  MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
	 ouput size: torch.Size([1, 64, 27, 27])
[3] forward  Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
	 ouput size: torch.Size([1, 192, 27, 27])
[4] forward  ReLU(inplace=True)
	 ouput size: torch.Size([1, 192, 27, 27])
[5] forward  MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
	 ouput size: torch.Size([1, 192, 13, 13])
[6] forward  Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
	 ouput size: torch.Size([1, 384, 13, 13])
[7] forward  ReLU(inplace=True)
	 ouput size: torch.Size([1, 384, 13, 13])
forwarded output size torch.Size([1, 384, 13, 13]) | switch indices : 2 | min : 0.0 | max : 34.77811050415039
*************************
[6] :  Con

NotImplementedError: The operator 'aten::max_unpool2d' is not currently implemented for the MPS device. If you want this op to be considered for addition please comment on https://github.com/pytorch/pytorch/issues/141287 and mention use-case, that resulted in missing op as well as commit hash Unknown. As a temporary fix, you can set the environment variable `PYTORCH_ENABLE_MPS_FALLBACK=1` to use the CPU as a fallback for this op. WARNING: this will be slower than running natively on MPS.